# Build AIC2026 BTC CLIP32 RVDB

This notebook builds `AIC2026_BTC_CLIP32_strict.rvdb` from precomputed BTC CLIP ViT-B/32 features.

Required input layout:
- `clip_features/clip-features-32/<video_id>.npy`
- `Keyframes/<video_id>/<local_keyframe>.jpg`
- `map-keyframes-aic25-b1/map-keyframes/<video_id>.csv`

The notebook uses strict mapping validation and stops if feature rows, JPG files, or mapping rows do not match.

In [ ]:
# Mount Google Drive. Put the repository and BTC artifacts in Drive first.
from google.colab import drive
drive.mount('/content/drive')

# Change this to the folder containing gui.py, unified_index.py, and the data folders.
PROJECT_ROOT = '/content/drive/MyDrive/AIC-AI-Challenge-main'

In [ ]:
# Install only the packages needed to build an RVDB from precomputed vectors.
%pip install -q 'numpy<2' 'h5py>=3.9,<4' 'lz4>=4.3,<5' 'faiss-cpu>=1.7.4' 'Pillow>=9.5,<12' 'opencv-python-headless>=4.8,<5'

In [ ]:
from pathlib import Path
import hashlib
import sys
import time
import numpy as np

PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

def first_existing(*paths):
    for path in paths:
        if path.exists():
            return path
    return paths[0]

FEATURES_DIR = first_existing(
    PROJECT_ROOT / 'clip_features' / 'clip-features-32',
    PROJECT_ROOT / 'clip_features' / 'clip_features_32'
)
KEYFRAMES_DIR = first_existing(
    PROJECT_ROOT / 'Keyframes',
    PROJECT_ROOT / 'keyframes'
)
MAP_DIR = first_existing(
    PROJECT_ROOT / 'map-keyframes-aic25-b1' / 'map-keyframes',
    PROJECT_ROOT / 'map-keyframes-aic25-b1'
)
OUTPUT_FILE = PROJECT_ROOT / 'output_keyframes' / 'AIC2026_BTC_CLIP32_strict.rvdb'
OVERWRITE = False

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FEATURES_DIR:', FEATURES_DIR)
print('KEYFRAMES_DIR:', KEYFRAMES_DIR)
print('MAP_DIR:', MAP_DIR)
print('OUTPUT_FILE:', OUTPUT_FILE)

In [ ]:
# Validate the three input artifacts before allocating the output file.
for name, path in {
    'features': FEATURES_DIR,
    'keyframes': KEYFRAMES_DIR,
    'mapping': MAP_DIR,
}.items():
    if not path.is_dir():
        raise FileNotFoundError(f'{name} directory not found: {path}')

feature_files = sorted(FEATURES_DIR.glob('*.npy'))
if not feature_files:
    raise FileNotFoundError(f'No .npy feature files found in {FEATURES_DIR}')

print(f'Found {len(feature_files):,} feature files')
missing_keyframes = []
missing_maps = []
total_rows = 0
for feature_file in feature_files:
    video_id = feature_file.stem
    keyframe_dir = KEYFRAMES_DIR / video_id
    map_file = MAP_DIR / f'{video_id}.csv'
    if not keyframe_dir.is_dir():
        missing_keyframes.append(video_id)
        continue
    if not map_file.is_file():
        missing_maps.append(video_id)
        continue
    array = np.load(feature_file, mmap_mode='r')
    if array.ndim != 2 or array.shape[1] != 512:
        raise ValueError(f'{video_id}: expected shape (N, 512), got {array.shape}')
    jpg_count = len(list(keyframe_dir.glob('*.jpg')))
    if jpg_count != array.shape[0]:
        raise ValueError(f'{video_id}: {array.shape[0]} feature rows but {jpg_count} JPG files')
    total_rows += array.shape[0]

if missing_keyframes or missing_maps:
    raise FileNotFoundError(f'Missing keyframes: {missing_keyframes[:10]}; missing mappings: {missing_maps[:10]}')

if OUTPUT_FILE.exists() and not OVERWRITE:
    raise FileExistsError(f'Output already exists: {OUTPUT_FILE}. Set OVERWRITE = True to replace it.')

print(f'Validated {total_rows:,} feature rows')
print('Strict mapping files and keyframe counts are present.')

In [ ]:
from unified_builder import UnifiedBuilderIntegration
from unified_index import UnifiedIndex, UnifiedIndexConfig

class NotebookLogger:
    def info(self, message, **kwargs):
        print('[INFO]', message)
    def warning(self, message, **kwargs):
        print('[WARN]', message)
    def error(self, message, **kwargs):
        print('[ERROR]', message)

logger = NotebookLogger()
builder = UnifiedBuilderIntegration.__new__(UnifiedBuilderIntegration)
builder.logger = logger
builder.system = None

def progress(current, message):
    print(f'[{current:3d}%] {message}')

started = time.time()
stats = builder.create_btc_clip_index_fast(
    features_dir=str(FEATURES_DIR),
    keyframes_dir=str(KEYFRAMES_DIR),
    map_dir=str(MAP_DIR),
    output_path=str(OUTPUT_FILE),
    progress_callback=progress,
    chunk_size=1000,
    clip_model_name='openai/clip-vit-base-patch32'
)
print(f'Build completed in {time.time() - started:.1f} seconds')
print(stats)

In [ ]:
# Verify the generated file and print the checksum used by the project docs.
if not OUTPUT_FILE.is_file():
    raise FileNotFoundError(OUTPUT_FILE)

sha256 = hashlib.sha256()
with OUTPUT_FILE.open('rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        sha256.update(block)

print('Output:', OUTPUT_FILE)
print('Size:', OUTPUT_FILE.stat().st_size, 'bytes')
print('SHA256:', sha256.hexdigest())
print('Expected production SHA256: 54fe6cc5507c51f29c5117601e87b50ae99dbe788ecb0997feba08b90cee594c')

## Use the generated file

Copy or leave the file in Drive, then open the GUI and choose **System Info -> Smart Load**. Select the generated `.rvdb` file.

The build requires the BTC `.npy` features, strict mapping CSVs, and matching JPG keyframes. It does not need the CLIP model because the vectors are already precomputed.